# 03 - Data Cleaning and Validation

This notebook converts OCR aggregate outputs into cleaned tables for analysis and dashboarding. The focus is on preserving the original extracted values while making validation status and imputed fields explicit.

## Main steps

1. Load station and vote outputs from `data/processed/`.
2. Repair selected missing ballot fields when they can be inferred from vote totals or related ballot counts.
3. Standardize constituency and party-list outputs into separate station and vote tables.
4. Remove physically impossible vote rows and duplicate vote records.
5. Create KNN-imputed vote tables for comparison with the raw OCR-based version.

## Validation rule

The core ballot check is:

$$	ext{ballots\_used} = 	ext{ballots\_good} + 	ext{ballots\_spoiled} + 	ext{ballots\_no\_vote}$$

Rows changed during cleaning keep a flag in `validation_flags`, so later analysis can distinguish observed OCR values from recovered or imputed values.

## Clean outputs

1. `5_18_station.csv`: polling-station metadata and ballot statistics for form 5/18.
2. `5_18_party_station.csv`: polling-station metadata and ballot statistics for form 5/18 (บช).
3. `5_18_votes.csv`: candidate-level constituency results.
4. `5_18_party_vote.csv`: party-level party-list results.
5. `5_18_votes_KNNImputed.csv` and `5_18_party_vote_KNNImputed.csv`: comparison datasets with missing vote cells imputed.


In [64]:
import pandas as pd
import numpy as np

In [65]:
stations_df = pd.read_csv("../data/processed/stations.csv")
votes_df = pd.read_csv("../data/processed/votes.csv")

In [66]:
col_to_drop = ['source_pdf', 'source_pages', 'ocr_status','ballots_received']
stations_df.drop(columns=col_to_drop, inplace=True)

In [67]:
stations_df.drop_duplicates(inplace=True)
stations_df.dropna(subset=['station_code'], inplace=True)

In [68]:
station = stations_df.copy()

In [69]:
stations_df.fillna({'ballots_no_vote': 0, 'ballots_spoiled': 0}, inplace=True)

In [70]:
def heal_election_data(stations_df, votes_df):
    # Estimate ballots_good from the parsed vote breakdown where needed.
    votes_sum = votes_df.groupby('station_code')['votes'].sum().reset_index()
    votes_sum.rename(columns={'votes': 'calculated_good_ballots'}, inplace=True)
    
    # Merge calculation back to stations
    df = stations_df.merge(votes_sum, on='station_code', how='left')
    
    # Fill missing ballots_good values with the calculated sum.
    mask_good = df['ballots_good'].isnull()
    df.loc[mask_good, 'ballots_good'] = df['calculated_good_ballots']
    df.loc[mask_good, 'validation_flags'] = df['validation_flags'].fillna('') + "|Recovered_Good_from_Votes"
    
    # Ballots Used = Good + Spoiled + No Vote
    mask_used = df['ballots_used'].isnull()
    df.loc[mask_used, 'ballots_used'] = (
        df['ballots_good'] + 
        df['ballots_spoiled'] + 
        df['ballots_no_vote']
    )
    df.loc[mask_used, 'validation_flags'] = df['validation_flags'].fillna('') + "|Calculated_Used_via_Sum"
    
    # Estimate voters_present from ballots_used when voters_present is missing.
    mask_present = df['voters_present'].isnull()
    df.loc[mask_present, 'voters_present'] = df['ballots_used']
    df.loc[mask_present, 'validation_flags'] = df['validation_flags'].fillna('') + "|Estimated_Present_from_Used"
    
    # Drop temporary helper columns.
    df.drop(columns=['calculated_good_ballots'], inplace=True)
    
    return df

# Apply ballot-field recovery rules.
stations_df = heal_election_data(stations_df, votes_df)


In [71]:
# Calculate the observed mean turnout for this constituency.
known_data = stations_df.dropna(subset=['eligible_voters', 'voters_present'])
constituency_turnout_rate = (known_data['voters_present'] / known_data['eligible_voters']).mean()

# If the observed mean is unavailable, use the documented fallback baseline.
final_rate = constituency_turnout_rate if not np.isnan(constituency_turnout_rate) else 0.71

# Apply the eligible-voter estimate.
mask = stations_df['eligible_voters'].isnull()
stations_df.loc[mask, 'eligible_voters'] = (stations_df['voters_present'] / final_rate).round()

# Tag estimated values for downstream dashboard review.
stations_df.loc[mask, 'validation_flags'] = (
    stations_df['validation_flags'].fillna('') + f"|ESTIMATED_ELIGIBLE_{final_rate:.2%}_TURNOUT"
)

In [72]:
stations_df.shape
stations_df.isnull().sum()

province             0
constituency_no      0
form_type            0
station_code         0
station_no           0
district             0
subdistrict          0
eligible_voters      0
voters_present       0
ballots_used         0
ballots_good         0
ballots_spoiled      0
ballots_no_vote      0
validation_flags    49
dtype: int64

In [73]:
import os

# Define the output directory under data/clean_data.
output_path = os.path.join('..', 'data', 'clean_data')

# Create the output directory if needed.
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"Directory created at: {os.path.abspath(output_path)}")

# Save station metadata for form 5/18.
station_file = os.path.join(output_path, '5_18_station.csv')
stations_df[stations_df['form_type'] == '5_18'].to_csv(station_file, index=False, encoding='utf-8-sig')

# Save station metadata for form 5/18 party-list.
party_file = os.path.join(output_path, '5_18_party_station.csv')
stations_df[stations_df['form_type'] == '5_18_party'].to_csv(party_file, index=False, encoding='utf-8-sig')

print("Files saved successfully in the root 'data/clean_data' folder.")

Files saved successfully in the root 'data/clean_data' folder.


In [74]:
station = (
    station
    .sort_values('ballots_good', ascending=False, na_position='last')
    .drop_duplicates(subset=['station_code'], keep='first')
    .reset_index(drop=True)
)


In [75]:
votes_df.drop_duplicates(inplace=True)
votes_df.dropna(subset=['station_code'], inplace=True)
votes_df.drop(columns=['votes_thai_word', 'needs_review'], inplace=True)

In [76]:
# Nullify physically impossible vote values for a station.
ballots_good_map = station.set_index('station_code')['ballots_good']
votes_df['_ballots_good'] = votes_df['station_code'].map(ballots_good_map)
impossible = (
    votes_df['votes'].notna() &
    votes_df['_ballots_good'].notna() &
    (votes_df['votes'] > votes_df['_ballots_good'])
)
print(f"Cleared {impossible.sum()} impossible vote values (votes > ballots_good)")
votes_df.loc[impossible, 'votes'] = np.nan
votes_df.drop(columns=['_ballots_good'], inplace=True)

# Deduplicate vote rows from the same station and entity.
# Keep the highest vote value for each station, entity, and form type.
before = len(votes_df)
votes_df = (
    votes_df
    .sort_values('votes', ascending=False, na_position='last')
    .drop_duplicates(subset=['station_code', 'entity_no', 'form_type'], keep='first')
    .reset_index(drop=True)
)
print(f"Removed {before - len(votes_df)} duplicate vote rows → {len(votes_df)} remaining")

Cleared 50 impossible vote values (votes > ballots_good)
Removed 1246 duplicate vote rows → 17941 remaining


In [77]:
votes_df.shape
votes_df.isnull().sum()

province               0
constituency_no        0
form_type              0
station_code           0
station_no             0
district               0
subdistrict            0
entity_kind            0
entity_no              0
entity_name            0
party_name         15939
votes               3548
dtype: int64

In [78]:
# 5_18_votes will contain constituency/candidate data
votes_5_18 = votes_df[votes_df['form_type'] == '5_18'].copy()

# 5_18_party_vote will contain party-list data
party_5_18 = votes_df[votes_df['form_type'] == '5_18_party'].copy()

party_5_18.drop(columns=['party_name'], inplace=True)

votes_5_18.to_csv(os.path.join(output_path, '5_18_votes.csv'), index=False, encoding='utf-8-sig')
party_5_18.to_csv(os.path.join(output_path, '5_18_party_vote.csv'), index=False, encoding='utf-8-sig')

print(f"Files saved to: {os.path.abspath(output_path)}")
print(f"5_18_votes rows: {len(votes_5_18)}")
print(f"5_18_party_vote rows: {len(party_5_18)}")

Files saved to: d:\DSDE\ProjectDSDE_election2026\data\clean_data
5_18_votes rows: 2002
5_18_party_vote rows: 15939


In [79]:
import pandas as pd
from sklearn.impute import KNNImputer

In [84]:
# Impute missing party-list vote values.

# 1. Load data
party_vote_df = pd.read_csv("../data/clean_data/5_18_party_vote.csv")
stations_df = pd.read_csv("../data/clean_data/5_18_station.csv")

# Merge station statistics after deduplicating station rows.
unique_stations = stations_df[['station_code', 'eligible_voters', 'ballots_good']].drop_duplicates(subset=['station_code'])

ml_party_df = party_vote_df.merge(
    unique_stations, 
    on='station_code', 
    how='left'
)

# Define features for KNN imputation.
features = ['entity_no', 'eligible_voters', 'ballots_good', 'votes']
train_data = ml_party_df[features]

print(f"Party votes missing BEFORE ML: {ml_party_df['votes'].isna().sum()}")

# Run KNN imputation.
imputer = KNNImputer(n_neighbors=10, weights='uniform')
imputed_data = imputer.fit_transform(train_data)

# Fill missing vote values with rounded imputed values.
predicted_votes = imputed_data[:, 3].round()
ml_party_df['votes'] = ml_party_df['votes'].fillna(pd.Series(predicted_votes))

print(f"Party votes missing AFTER ML: {ml_party_df['votes'].isna().sum()}")

# Save the imputed dataset.
ml_party_df = ml_party_df.drop(columns=['eligible_voters', 'ballots_good'])
ml_party_df.to_csv("../data/clean_data/5_18_party_vote_KNNImputed.csv", index=False)
print("Saved KNN-imputed party votes to: ../data/clean_data/5_18_party_vote_KNNImputed.csv\n")

Party votes missing BEFORE ML: 3423
Party votes missing AFTER ML: 0
Saved KNN-imputed party votes to: ../data/clean_data/5_18_party_vote_KNNImputed.csv



In [85]:
# Impute missing constituency candidate vote values.

# 1. Load data
candidate_vote_df = pd.read_csv("../data/clean_data/5_18_votes.csv")
# Reuse the deduplicated station table from the party-list imputation step.

# Merge station statistics using the deduplicated station table.
ml_candidate_df = candidate_vote_df.merge(
    unique_stations, 
    on='station_code', 
    how='left'
)

# Define features for KNN imputation.
features = ['entity_no', 'eligible_voters', 'ballots_good', 'votes']
train_data = ml_candidate_df[features]

print(f"Candidate votes missing BEFORE ML: {ml_candidate_df['votes'].isna().sum()}")

# Run KNN imputation.
imputer = KNNImputer(n_neighbors=10, weights='uniform')
imputed_data = imputer.fit_transform(train_data)

# Fill missing vote values with rounded imputed values.
predicted_votes = imputed_data[:, 3].round()
ml_candidate_df['votes'] = ml_candidate_df['votes'].fillna(pd.Series(predicted_votes))

print(f"Candidate votes missing AFTER ML: {ml_candidate_df['votes'].isna().sum()}")

# Save the imputed dataset.
ml_candidate_df = ml_candidate_df.drop(columns=['eligible_voters', 'ballots_good'])
ml_candidate_df.to_csv("../data/clean_data/5_18_votes_KNNImputed.csv", index=False)
print("Saved KNN-imputed candidate votes to: ../data/clean_data/5_18_votes_KNNImputed.csv")

Candidate votes missing BEFORE ML: 125
Candidate votes missing AFTER ML: 0
Saved KNN-imputed candidate votes to: ../data/clean_data/5_18_votes_KNNImputed.csv
